#Import Libraries & Data

In [41]:
#import libraries
import pandas as pd
import pandas_gbq
import pydata_google_auth
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

import matplotlib.style as style
style.available
style.use('fivethirtyeight')

##Authenticate

In [42]:
import pandas_gbq
import pydata_google_auth

SCOPES = [
    'https://www.googleapis.com/auth/cloud-platform',
    'https://www.googleapis.com/auth/drive',
]

# get credentials
credentials = pydata_google_auth.get_user_credentials(
    SCOPES,
    auth_local_webserver=False)

# AAL — Ever-modified vintage monitoring

Monthly **ever-modified** (`flag_modified`) analysis by **origination vintage**, with **`mod_start_date`** for months-on-book at modification.

**Run order:** Authenticate → Config → BigQuery pull → Helpers → **Analytics (A–G)**.

FPD segmentation and first-pay monitoring remain in `AAL_First_Pay_Default_2026.ipynb` only.


In [ ]:
# --- Ever-modified vintage monitoring configuration ---
import pandas as pd

MODEL_ERA_START = '2026-01-01'
VINTAGE_COL = 'loan_vintage_month'
TIER_COL = 'borrower_origination_risk_group'
MODIFIED_FLAG_COL = 'flag_modified'
MOD_START_DATE_COL = 'mod_start_date'

MODIFICATION_MIN_DAYS_ON_BOOK = 120
MODIFICATION_REFERENCE_VINTAGE = '2026-01'
MODIFICATION_TRAILING_VINTAGES = 6
MIN_PCT_MATURE_FOR_CI = 0.80
PSI_MIN_PCT_MATURE = 0.80
MOB_DAYS_PER_MONTH = 30

AS_OF_DATE = pd.Timestamp.today().normalize()
NOTEBOOK_REV = "2026-09-mod-standalone-v1"

print(
    f"Config: MODEL_ERA_START={MODEL_ERA_START}, MODIFIED_FLAG_COL={MODIFIED_FLAG_COL}, "
    f"MOD_START_DATE_COL={MOD_START_DATE_COL}, MODIFICATION_MIN_DAYS_ON_BOOK={MODIFICATION_MIN_DAYS_ON_BOOK}, "
    f"AS_OF_DATE={AS_OF_DATE.date()}, NOTEBOOK_REV={NOTEBOOK_REV}"
)


In [ ]:
post_origination_cols = [
    'flag_modified',
    'mod_start_date',
    'flag_1st_pay_default',
    'days_elapsed_origination_to_dq',
    'days_elapsed_origination_to_dq_30',
    'count_dpd_30',
    'count_dpd_1_to_10',
    'flag_dq',
    'loan_prepaid_flag',
    'loan_1st_pay_default_return_code',
    'count_dpd_cured',
    'loan_no_payment_straight_to_chargeoff_flag',
    'loan_chargeoff_flag',
    'loan_contractual_chargeoff_flag',
    'flag_never_pay',
    'principal_default_percent',
    'loan_only_one_transaction_day_of_week',
    'loan_in_debt_settlement_flag',
    'loan_deceased_flag',
    'loan_total_overall_payment_amount',
]


In [ ]:
sql = f"""
select *
from `ffam-data-platform-loan-ops.report_views.v_aal_early_pay_default`
where DATE(loan_vintage_month) >= DATE('{MODEL_ERA_START}')
"""

df = pandas_gbq.read_gbq(
    sql,
    project_id='ffn-dw-bigquery-prd',
    credentials=credentials,
    dialect='standard',
)
print(f"Loaded {len(df):,} rows | vintages from {MODEL_ERA_START}")
df.head()


## Business guide

| Question | Section |
|----------|--------|
| Which vintages have higher **ever-modified %**? | **A**, **C** |
| Are **recent months** above earlier vintages? | **B** (bps vs baseline) |
| **Tier** concentration within a vintage? | **D** heatmap |
| **Mix** vs **within-tier rate**? | **E** decomposition |
| **Booking mix** shift vs reference? | **F** PSI |
| **When** mods occur (MOB)? | **G** `mod_start_date` |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def _vintage_period(series):
    """Normalize loan_vintage_month to pandas Period[M]."""
    s = pd.to_datetime(series)
    return s.dt.to_period('M')

def _period_to_str(p):
    if isinstance(p, pd.Period):
        return str(p)
    return pd.Period(p, freq='M').strftime('%Y-%m')

def tier_mix_and_rates(df_m, vintage_col, tier_col, target):
    """Tier mix (row %) and within-tier FPD rate by vintage."""
    v = _vintage_period(df_m[vintage_col])
    tmp = df_m.assign(_vintage=v)
    mix = pd.crosstab(tmp['_vintage'], tmp[tier_col], normalize='index')
    rates = tmp.groupby(['_vintage', tier_col], observed=True)[target].mean().unstack(tier_col)
    # Coerce to numeric + string labels for charts/heatmaps (Period index -> object dtype)
    if rates is not None and not rates.empty:
        rates.index = [_period_to_str(x) for x in rates.index]
        rates.columns = [str(c) for c in rates.columns]
        rates = rates.apply(pd.to_numeric, errors='coerce')
    if mix is not None and not mix.empty:
        mix.index = [_period_to_str(x) for x in mix.index]
        mix.columns = [str(c) for c in mix.columns]
        mix = mix.apply(pd.to_numeric, errors='coerce')
    return mix, rates

def fpd_decomposition(df_a, df_b, tier_col, target):
    """
    Portfolio FPD change decomposition (tier = risk segment).
    ΔFPD = mix_effect + rate_effect + interaction
      mix_effect    = Σ_t (p_B - p_A) * r_A
      rate_effect   = Σ_t p_B * (r_B - r_A)
      interaction   = Σ_t (p_B - p_A) * (r_B - r_A)
    where p_t = tier mix, r_t = within-tier FPD rate.
    """
    def _vec(d):
        p = d[tier_col].value_counts(normalize=True)
        r = d.groupby(tier_col, observed=True)[target].mean()
        tiers = sorted(set(p.index) | set(r.index), key=lambda x: str(x))
        p = p.reindex(tiers, fill_value=0.0)
        r = r.reindex(tiers, fill_value=0.0)
        return p, r, tiers

    p_a, r_a, tiers = _vec(df_a)
    p_b, r_b, _ = _vec(df_b)
    mix_eff = ((p_b - p_a) * r_a).sum()
    rate_eff = (p_b * (r_b - r_a)).sum()
    interact = ((p_b - p_a) * (r_b - r_a)).sum()
    fpd_a = df_a[target].mean()
    fpd_b = df_b[target].mean()
    return {
        'fpd_a': fpd_a,
        'fpd_b': fpd_b,
        'delta_fpd': fpd_b - fpd_a,
        'mix_effect': mix_eff,
        'rate_effect': rate_eff,
        'interaction': interact,
        'tiers': tiers,
        'p_a': p_a,
        'p_b': p_b,
        'r_a': r_a,
        'r_b': r_b,
    }

def psi_categorical(expected: pd.Series, actual: pd.Series):
    """PSI on categorical bins (tier or binned feature)."""
    e = expected.value_counts(normalize=True)
    a = actual.value_counts(normalize=True)
    bins = sorted(set(e.index) | set(a.index), key=lambda x: str(x))
    e = e.reindex(bins, fill_value=0.0)
    a = a.reindex(bins, fill_value=0.0)
    eps = 1e-6
    e = e.clip(lower=eps)
    a = a.clip(lower=eps)
    return float(((a - e) * np.log(a / e)).sum())

def psi_numeric(expected: pd.Series, actual: pd.Series, n_bins=10):
    """PSI on numeric feature using quantile bins fit on expected (reference) sample."""
    exp = pd.to_numeric(expected, errors='coerce').dropna()
    act = pd.to_numeric(actual, errors='coerce').dropna()
    if len(exp) < n_bins or len(act) == 0:
        return np.nan
    try:
        cuts = pd.qcut(exp, q=n_bins, duplicates='drop')
    except ValueError:
        return np.nan
    bins = cuts.cat.categories
    exp_b = pd.cut(exp, bins=bins).value_counts(normalize=True)
    act_b = pd.cut(act, bins=bins).value_counts(normalize=True)
    idx = sorted(set(exp_b.index) | set(act_b.index), key=str)
    e = exp_b.reindex(idx, fill_value=0.0).clip(lower=1e-6)
    a = act_b.reindex(idx, fill_value=0.0).clip(lower=1e-6)
    return float(((a - e) * np.log(a / e)).sum())

def psi_flag(val):
    if pd.isna(val):
        return 'n/a'
    if val > 0.25:
        return 'material shift (>0.25)'
    if val > 0.1:
        return 'moderate shift (>0.10)'
    return 'stable (<=0.10)'

def wilson_ci(successes, n, z=1.96):
    """Wilson score interval for a binomial proportion."""
    if n is None or n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))

def cohort_table_with_ci(
    df_mon,
    target,
    mature_col='is_mature_fpd',
    event_count_col='fpd_count',
    rate_col='fpd_rate',
):
    """Build cohort metrics with Wilson 95% CI on mature-loan outcome rate."""
    rows = []
    for v in sorted(df_mon['_vintage'].unique()):
        g_all = df_mon[df_mon['_vintage'] == v]
        g_mat = g_all[g_all[mature_col]]
        n_mat = len(g_mat)
        fpd = int(g_mat[target].sum()) if n_mat else 0
        rate = g_mat[target].mean() if n_mat else np.nan
        lo, hi = wilson_ci(fpd, n_mat) if n_mat else (np.nan, np.nan)
        row = {
            'vintage_month': _period_to_str(v),
            'loan_count': len(g_all),
            'mature_loan_count': n_mat,
            'pct_mature': g_all[mature_col].mean(),
        }
        row[event_count_col] = fpd
        row[rate_col] = rate
        row[rate_col + '_ci_low'] = lo
        row[rate_col + '_ci_high'] = hi
        rows.append(row)
    return pd.DataFrame(rows)

def _coerce_fpd_rate_frame(rate_df):
    """Force vintage x tier FPD rates to float (handles Period index / object cells)."""
    if rate_df is None or rate_df.empty:
        return rate_df
    out = rate_df.copy()
    out.index = [_period_to_str(x) if isinstance(x, pd.Period) else str(x) for x in out.index]
    out.columns = [str(c) for c in out.columns]
    try:
        stacked = out.stack(future_stack=True)
    except TypeError:
        stacked = out.stack()
    numeric = pd.to_numeric(stacked, errors='coerce')
    wide = numeric.unstack()
    return wide.astype(float)

def plot_tier_fpd_heatmap(tier_fpd_rates, title='Within-tier FPD rate by vintage (mature loans)'):
    if tier_fpd_rates is None or tier_fpd_rates.empty:
        print('Tier FPD heatmap skipped.')
        return
    wide = _coerce_fpd_rate_frame(tier_fpd_rates)
    # tiers on rows, vintages on columns (presentation)
    data = (wide.T * 100).astype(float)
    fig, ax = plt.subplots(figsize=(14, max(4, 0.35 * data.shape[0])))
    sns.heatmap(data, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'FPD %'})
    ax.set_title(title)
    ax.set_xlabel('Vintage month')
    ax.set_ylabel('Risk tier')
    plt.tight_layout()
    plt.show()

def plot_psi_heatmap(psi_table, title='PSI vs reference vintage'):
    if psi_table is None or psi_table.empty:
        print('PSI heatmap skipped.')
        return
    pivot = psi_table.pivot(index='feature', columns='vintage_month', values='psi')
    fig, ax = plt.subplots(figsize=(12, max(4, 0.4 * pivot.shape[0])))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='Blues', ax=ax, cbar_kws={'label': 'PSI'})
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

def _psi_numeric_feature_cols(df, post_origination_cols, max_cols=5):
    """Shared PSI numeric feature list (pre-origination only)."""
    candidates = [
        'loan_application_PTI_ratio',
        'loan_application_NDI_ratio',
        'fdr_vs_aal_payment_variance',
        'loan_origination_loan_amount',
        'days_elapsed_negotiation',
        'avg_monthly_draft',
    ]
    cols = [c for c in candidates if c in df.columns and c not in post_origination_cols]
    return cols[:max_cols]

def _vintage_pct_mature(df_mon, vintage_period):
    g = df_mon[df_mon['_vintage'] == vintage_period]
    if g.empty:
        return 0.0
    return float(_pct_mature_fraction(g['is_mature_fpd'].mean()))

def add_modification_observability(
    df_in,
    as_of_date,
    min_days_on_book,
    vintage_col,
    mod_start_date_col=None,
    modified_flag_col=None,
):
    """
    Loans need enough time on book before ever-modified rates are comparable across vintages.
    Uses vintage month start vs AS_OF_DATE (same vintage-month grain as Part B).

    When `mod_start_date_col` is present, also computes days/MOB from vintage start to modification.
    """
    work = df_in.copy()
    if '_vintage' not in work.columns:
        work['_vintage'] = _vintage_period(work[vintage_col])
    as_of = pd.Timestamp(as_of_date).normalize()
    vintage_start = work['_vintage'].apply(lambda p: p.to_timestamp())
    work['days_since_vintage_start'] = (as_of - vintage_start).dt.days
    work['is_mature_mod'] = work['days_since_vintage_start'] >= int(min_days_on_book)

    if mod_start_date_col and mod_start_date_col in work.columns:
        work['_mod_start'] = pd.to_datetime(work[mod_start_date_col], errors='coerce')
        work['days_vintage_to_mod'] = (work['_mod_start'] - vintage_start).dt.days
        mob_days = globals().get('MOB_DAYS_PER_MONTH', 30)
        work['mob_at_mod'] = work['days_vintage_to_mod'] / float(mob_days)
        if modified_flag_col and modified_flag_col in work.columns:
            flagged = work[modified_flag_col].astype(float).fillna(0) > 0
            has_date = work['_mod_start'].notna()
            mismatch = flagged ^ has_date
            if mismatch.any():
                print(
                    f"Note: {int(mismatch.sum())} loans where {modified_flag_col} and "
                    f"{mod_start_date_col} disagree — review view logic."
                )
    return work

def build_modification_timing_by_vintage(df_mod, modified_flag_col, mature_col='is_mature_mod'):
    """Median MOB-at-modification by vintage (modified loans only, mature-enough cohort)."""
    d = df_mod[df_mod[mature_col]].copy()
    if 'mob_at_mod' not in d.columns:
        return pd.DataFrame()
    mod = d[d[modified_flag_col].astype(float).fillna(0) > 0].dropna(subset=['mob_at_mod'])
    if mod.empty:
        return pd.DataFrame()
    rows = []
    for v in sorted(mod['_vintage'].unique()):
        g = mod[mod['_vintage'] == v]
        g_all = d[d['_vintage'] == v]
        rows.append({
            'vintage_month': _period_to_str(v),
            'modified_loan_count': len(g),
            'mature_loan_count': len(g_all),
            'modified_rate': g_all[modified_flag_col].mean(),
            'median_mob_at_mod': g['mob_at_mod'].median(),
            'p25_mob_at_mod': g['mob_at_mod'].quantile(0.25),
            'p75_mob_at_mod': g['mob_at_mod'].quantile(0.75),
            'median_days_vintage_to_mod': g['days_vintage_to_mod'].median(),
        })
    return pd.DataFrame(rows)

def plot_vintage_outcome_trend(
    cohort_ci,
    rate_col='fpd_rate',
    title='Vintage outcome rate (mature loans, 95% CI)',
    min_pct_mature_for_ci=None,
):
    """Same layout as FPD trend; parameterized rate column."""
    min_pct = min_pct_mature_for_ci
    if min_pct is None:
        min_pct = globals().get('MIN_PCT_MATURE_FOR_CI', 0.80)
    plot_df = cohort_ci.copy().sort_values('vintage_month')
    if plot_df.empty:
        print('Vintage outcome trend skipped — no vintages.')
        return
    lo_col, hi_col = rate_col + '_ci_low', rate_col + '_ci_high'
    x = np.arange(len(plot_df))
    y = plot_df[rate_col].astype(float) * 100
    y_plot = y.copy()
    ylo = plot_df[lo_col].astype(float) * 100
    yhi = plot_df[hi_col].astype(float) * 100
    pct_m = plot_df['pct_mature'].map(_pct_mature_fraction).fillna(0).astype(float)
    ci_ok = pct_m >= float(min_pct)
    fig, ax1 = plt.subplots(figsize=(12, 5))
    if ci_ok.any():
        ax1.plot(x[ci_ok], y_plot[ci_ok], 'o-', color='#2ca02c', label='Rate % (mature enough for CI)')
        _y_ok = y[ci_ok].astype(float)
        _ylo_ok = ylo[ci_ok].astype(float)
        _yhi_ok = yhi[ci_ok].astype(float)
        _x_ok = x[ci_ok]
        _err_lo = np.clip(_y_ok - _ylo_ok, 0, None)
        _err_hi = np.clip(_yhi_ok - _y_ok, 0, None)
        _valid = np.isfinite(_y_ok) & np.isfinite(_err_lo) & np.isfinite(_err_hi)
        if _valid.any():
            ax1.errorbar(
                _x_ok[_valid],
                _y_ok[_valid],
                yerr=[_err_lo[_valid], _err_hi[_valid]],
                fmt='none', ecolor='#98df8a', elinewidth=1.5, capsize=3, alpha=0.45,
        )
    if (~ci_ok).any():
        y_plot = y_plot.copy()
        y_plot[~ci_ok] = np.nan
        ax1.plot(x[~ci_ok], y_plot[~ci_ok], 'o', color='#bdbdbd', linestyle='None',
                 label='Rate % (immature — CI hidden)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(plot_df['vintage_month'], rotation=45, ha='right')
    ax1.set_ylabel('Rate (%)')
    ax1.set_title(title)
    ax2 = ax1.twinx()
    ax2.bar(x, plot_df['loan_count'], alpha=0.2, color='gray', label='Loan count (all)')
    ax2.set_ylabel('Loan count')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

def build_modification_vintage_attribution(cohort_ci, rate_col, trailing_n, baseline_exclude_trailing=True):
    """
    Rank recent vintages vs baseline (earlier mature vintages) on modified rate (bps).
    """
    if cohort_ci is None or cohort_ci.empty:
        return pd.DataFrame()
    d = cohort_ci.dropna(subset=[rate_col]).sort_values('vintage_month').copy()
    if d.empty:
        return pd.DataFrame()
    trail = d.tail(int(trailing_n))
    if baseline_exclude_trailing and len(d) > len(trail):
        base = d.iloc[: -len(trail)]
    else:
        base = d.iloc[: max(1, len(d) - 1)]
    base_rate = float(base[rate_col].mean()) if len(base) else np.nan
    out = trail.copy()
    out['baseline_rate'] = base_rate
    out['bps_vs_baseline'] = (out[rate_col] - base_rate) * 10000
    out['share_of_loans_pct'] = out['loan_count'] / out['loan_count'].sum() * 100
    return out.sort_values('bps_vs_baseline', ascending=False)

def plot_decomposition_waterfall(decomp, label='Portfolio FPD change'):
    if decomp is None:
        print('Decomposition waterfall skipped.')
        return
    mix_bps = decomp['mix_effect'] * 10000
    rate_bps = decomp['rate_effect'] * 10000
    interact_bps = decomp['interaction'] * 10000
    total_bps = decomp['delta_fpd'] * 10000
    start_bps = decomp['fpd_a'] * 10000
    end_bps = decomp['fpd_b'] * 10000
    labels = ['Mix', 'Rate', 'Interaction']
    deltas = [mix_bps, rate_bps, interact_bps]
    colors = ['#2ca02c', '#d62728', '#9467bd']
    fig, ax = plt.subplots(figsize=(9, 5))
    running = start_bps
    ax.bar(0, start_bps, color='#9edae5', label='Prior portfolio FPD (bps)')
    for i, (lab, d) in enumerate(zip(labels, deltas), start=1):
        bottom = running if d >= 0 else running + d
        ax.bar(i, abs(d), bottom=bottom, color=colors[i - 1], label=f'{lab} ({d:+.1f} bps)')
        running += d
    ax.bar(len(labels) + 1, end_bps, color='#1f77b4', label='Current portfolio FPD (bps)')
    ax.set_xticks(range(len(labels) + 2))
    ax.set_xticklabels(['Prior'] + labels + ['Current'])
    ax.set_ylabel('FPD (bps)')
    ax.set_title(f'{label}: net change {total_bps:+.1f} bps')
    ax.legend(loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

def _pct_mature_fraction(val):
    """Normalize pct_mature to 0-1 whether stored as fraction or percent."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return np.nan
    v = float(val)
    return v / 100.0 if v > 1.5 else v



## Run analytics (sections A–G)

In [ ]:
# --- Analytics: ever-modified by vintage ---
if MODIFIED_FLAG_COL not in df.columns:
    raise KeyError(f"{MODIFIED_FLAG_COL!r} missing — re-run BigQuery after view deploy.")

df_mon = df.copy()
df_mon['_vintage'] = _vintage_period(df_mon[VINTAGE_COL])
era = pd.Period(pd.Timestamp(MODEL_ERA_START), freq='M')
df_mon = df_mon[df_mon['_vintage'] >= era].copy()

df_mod = add_modification_observability(
    df_mon,
    AS_OF_DATE,
    MODIFICATION_MIN_DAYS_ON_BOOK,
    VINTAGE_COL,
    mod_start_date_col=MOD_START_DATE_COL if MOD_START_DATE_COL in df_mon.columns else None,
    modified_flag_col=MODIFIED_FLAG_COL,
)
df_mod_mature = df_mod[df_mod['is_mature_mod']].copy()
mod_mature_vintages = sorted(df_mod_mature['_vintage'].unique())

mod_cohort_ci = cohort_table_with_ci(
    df_mod,
    MODIFIED_FLAG_COL,
    mature_col='is_mature_mod',
    event_count_col='modified_count',
    rate_col='modified_rate',
)

print("### A. Ever-modified cohort table (by vintage month)")
display(mod_cohort_ci.assign(
    modified_rate=lambda x: (x['modified_rate'] * 100).round(3),
    pct_mature=lambda x: (x['pct_mature'].map(_pct_mature_fraction) * 100).round(1),
))

mod_attrib = build_modification_vintage_attribution(
    mod_cohort_ci, 'modified_rate', MODIFICATION_TRAILING_VINTAGES,
)
print(f"### B. Vintage attribution — last {MODIFICATION_TRAILING_VINTAGES} months vs earlier baseline")
if mod_attrib.empty:
    print("Not enough mature vintages for attribution.")
else:
    display(mod_attrib.assign(
        modified_rate=lambda d: (d['modified_rate'] * 100).round(3),
        baseline_rate=lambda d: (d['baseline_rate'] * 100).round(3),
        bps_vs_baseline=lambda d: d['bps_vs_baseline'].round(1),
        share_of_loans_pct=lambda d: d['share_of_loans_pct'].round(1),
    ))

print("### C. Ever-modified rate trend + volume")
plot_vintage_outcome_trend(
    mod_cohort_ci,
    rate_col='modified_rate',
    title='Ever-modified rate by vintage (mature-enough loans, 95% CI)',
    min_pct_mature_for_ci=MIN_PCT_MATURE_FOR_CI,
)

if mod_mature_vintages:
    _, tier_mod_rates = tier_mix_and_rates(df_mod_mature, VINTAGE_COL, TIER_COL, MODIFIED_FLAG_COL)
    print("### D. Within-tier ever-modified rate heatmap")
    plot_tier_fpd_heatmap(
        tier_mod_rates,
        title='Within-tier ever-modified rate by vintage (mature-enough loans)',
    )

if len(mod_mature_vintages) >= 2:
    mod_latest, mod_prior = mod_mature_vintages[-1], mod_mature_vintages[-2]
    decomp_mom = fpd_decomposition(
        df_mod_mature[df_mod_mature['_vintage'] == mod_prior],
        df_mod_mature[df_mod_mature['_vintage'] == mod_latest],
        TIER_COL,
        MODIFIED_FLAG_COL,
    )
    print(f"### E. Mix vs rate: {_period_to_str(mod_prior)} → {_period_to_str(mod_latest)}")
    display(pd.DataFrame([{
        'modified_bps_change': round(decomp_mom['delta_fpd'] * 10000, 1),
        'mix_effect_bps': round(decomp_mom['mix_effect'] * 10000, 1),
        'rate_effect_bps': round(decomp_mom['rate_effect'] * 10000, 1),
        'interaction_bps': round(decomp_mom['interaction'] * 10000, 1),
    }]))
    plot_decomposition_waterfall(
        decomp_mom,
        label=f"{_period_to_str(mod_prior)} → {_period_to_str(mod_latest)} (ever-modified)",
    )
    mod_ref = (
        pd.Period(MODIFICATION_REFERENCE_VINTAGE, freq='M')
        if MODIFICATION_REFERENCE_VINTAGE else mod_mature_vintages[0]
    )
    if mod_ref in mod_mature_vintages and mod_ref != mod_latest:
        decomp_ref = fpd_decomposition(
            df_mod_mature[df_mod_mature['_vintage'] == mod_ref],
            df_mod_mature[df_mod_mature['_vintage'] == mod_latest],
            TIER_COL,
            MODIFIED_FLAG_COL,
        )
        print(f"### E2. vs reference {_period_to_str(mod_ref)}")
        display(pd.DataFrame([{
            'modified_bps_change': round(decomp_ref['delta_fpd'] * 10000, 1),
            'mix_effect_bps': round(decomp_ref['mix_effect'] * 10000, 1),
            'rate_effect_bps': round(decomp_ref['rate_effect'] * 10000, 1),
            'interaction_bps': round(decomp_ref['interaction'] * 10000, 1),
        }]))

psi_table = pd.DataFrame()
mod_ref = (
    pd.Period(MODIFICATION_REFERENCE_VINTAGE, freq='M')
    if MODIFICATION_REFERENCE_VINTAGE else (mod_mature_vintages[0] if mod_mature_vintages else None)
)
if mod_ref is not None and mod_ref in df_mod['_vintage'].unique():
    mature_enough = set(
        mod_cohort_ci.loc[
            mod_cohort_ci['pct_mature'].map(_pct_mature_fraction) >= PSI_MIN_PCT_MATURE,
            'vintage_month',
        ].astype(str)
    )
    ref_df = df_mod_mature[df_mod_mature['_vintage'] == mod_ref]
    psi_rows = []
    for v in mod_mature_vintages:
        if v == mod_ref or _period_to_str(v) not in mature_enough:
            continue
        cur_df = df_mod_mature[df_mod_mature['_vintage'] == v]
        tpsi = psi_categorical(ref_df[TIER_COL], cur_df[TIER_COL])
        psi_rows.append({
            'vintage_month': _period_to_str(v),
            'feature': TIER_COL,
            'psi': round(tpsi, 4),
            'flag': psi_flag(tpsi),
        })
        for col in _psi_numeric_feature_cols(df_mod_mature, post_origination_cols):
            val = psi_numeric(ref_df[col], cur_df[col])
            psi_rows.append({
                'vintage_month': _period_to_str(v),
                'feature': col,
                'psi': round(val, 4) if pd.notna(val) else np.nan,
                'flag': psi_flag(val),
            })
    if psi_rows:
        psi_table = pd.DataFrame(psi_rows)
        print(f"### F. PSI vs reference {_period_to_str(mod_ref)}")
        display(psi_table.sort_values(['vintage_month', 'feature']))
        plot_psi_heatmap(psi_table, title=f'PSI vs reference {_period_to_str(mod_ref)}')

if MOD_START_DATE_COL in df_mod.columns and 'mob_at_mod' in df_mod.columns:
    mod_timing = build_modification_timing_by_vintage(df_mod, MODIFIED_FLAG_COL)
    print("### G. Time to modification (MOB at mod_start_date)")
    if mod_timing.empty:
        print("No modified loans with mod_start_date in mature cohorts.")
    else:
        display(mod_timing.assign(
            modified_rate=lambda d: (d['modified_rate'] * 100).round(3),
            median_mob_at_mod=lambda d: d['median_mob_at_mod'].round(2),
            p25_mob_at_mod=lambda d: d['p25_mob_at_mod'].round(2),
            p75_mob_at_mod=lambda d: d['p75_mob_at_mod'].round(2),
            median_days_vintage_to_mod=lambda d: d['median_days_vintage_to_mod'].round(0),
        ))
else:
    print(f"### G. Skipped — {MOD_START_DATE_COL!r} not in pull.")

if not mod_attrib.empty:
    hot = mod_attrib.sort_values('bps_vs_baseline', ascending=False).iloc[0]
    print(
        f"\n**Summary:** Top recent vintage vs baseline: {hot['vintage_month']} "
        f"({hot['bps_vs_baseline']:+.0f} bps, {hot['modified_rate']*100:.2f}% ever-modified)."
    )
